# 이미지 회귀 (Image Regression) — PetFinder Pawpularity (PyTorch) — Colab

이미지에서 **연속 점수**를 예측하는 회귀 기본 예제입니다 (분류가 아닌 회귀).

- 핵심 흐름: 사진 → 사전학습 ResNet → **1개 연속값**(Pawpularity 0~100) 회귀, RMSE 평가.
- IOAI 2025 **Weather(위성영상→강수량 예측)** 처럼 *이미지→연속값* 예측의 기반 기법입니다. (Weather 공식 해법은 U-Net 으로 연속 맵을 회귀 — 본 노트북은 스칼라 회귀 기본기)
- [PetFinder Pawpularity](https://www.kaggle.com/c/petfinder-pawpularity-score) 사용. 토큰이 **노트북에 내장**되어 바로 실행됩니다.

> ⚙️ **GPU 권장**.  
> ⚠️ **사전 필수**: `jinusun` 계정으로 [대회 규칙](https://www.kaggle.com/c/petfinder-pawpularity-score/rules) Join 필요(안 하면 403).  
> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부 공유 금지.

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "torchvision", "numpy", "pandas", "pillow"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 PetFinder 데이터 다운로드

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("petfinder-pawpularity-score", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
print("내용:", [f for f in sorted(os.listdir(WORK_DIR)) if not f.startswith(".")][:10])

## 2. Dataset / DataLoader

`train.csv` 의 `Id`, `Pawpularity`(0~100) 를 쓰고 이미지는 `train/<Id>.jpg`. 타깃은 0~1 로 스케일합니다.

In [ ]:
import numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

df = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
IMG_DIR = os.path.join(WORK_DIR, "train")
MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
train_tf = transforms.Compose([transforms.Resize((224,224)), transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(), transforms.Normalize(MEAN,STD)])
eval_tf = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), transforms.Normalize(MEAN,STD)])

class PetData(Dataset):
    def __init__(self, frame, tf, train=True):
        self.frame = frame.reset_index(drop=True); self.tf = tf; self.train = train
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        img = self.tf(Image.open(os.path.join(IMG_DIR, row["Id"]+".jpg")).convert("RGB"))
        if self.train:
            return img, torch.tensor(row["Pawpularity"]/100.0, dtype=torch.float32)
        return img, row["Id"]

g = np.random.RandomState(42); idx = g.permutation(len(df)); cut=int(len(df)*0.9)
tr_df, va_df = df.iloc[idx[:cut]], df.iloc[idx[cut:]]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader = DataLoader(PetData(tr_df, train_tf), batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(PetData(va_df, eval_tf), batch_size=32, num_workers=2)
print("device:", device, "| train:", len(tr_df), "val:", len(va_df))

## 3. ResNet18 회귀 헤드

In [ ]:
import torch.nn as nn
from torchvision import models

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 1)  # 회귀: 출력 1개
model = model.to(device)
print("ResNet18 회귀 모델 준비 완료")

## 4. 학습 (MSE) & RMSE 평가

출력에 sigmoid 를 씌워 0~1 로 제한하고, 평가 시 ×100 하여 원 스케일 RMSE 를 봅니다.

In [ ]:
crit = nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-4)
EPOCHS = 3
for epoch in range(1, EPOCHS+1):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = torch.sigmoid(model(xb)).squeeze(1)
        opt.zero_grad(); crit(pred, yb).backward(); opt.step()
    model.eval(); se=0.0; n=0
    with torch.no_grad():
        for xb, yb in val_loader:
            p = torch.sigmoid(model(xb.to(device))).squeeze(1).cpu()
            se += (((p - yb)*100)**2).sum().item(); n += len(yb)
    print(f"epoch {epoch} | val RMSE {(se/n)**0.5:.3f}")

## 5. 테스트 예측 & 제출 파일 생성

In [ ]:
test_df = pd.read_csv(os.path.join(WORK_DIR, "test.csv"))
TEST_DIR = os.path.join(WORK_DIR, "test")
eval_tf2 = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), transforms.Normalize(MEAN,STD)])

model.eval(); preds=[]
with torch.no_grad():
    for _, row in test_df.iterrows():
        img = eval_tf2(Image.open(os.path.join(TEST_DIR, row["Id"]+".jpg")).convert("RGB")).unsqueeze(0).to(device)
        preds.append(float(torch.sigmoid(model(img)).item()*100))

submission_path = os.path.join(WORK_DIR, "submission.csv")
sub = pd.DataFrame({"Id": test_df["Id"], "Pawpularity": preds})
sub.to_csv(submission_path, index=False)
print("Saved:", submission_path); sub.head()

## 6. 제출 파일 내려받기

In [ ]:
try:
    from google.colab import files; files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e, "| 위치:", submission_path)

## 🏁 제출 안내

위에서 생성된 `submission.csv` 를 아래에서 제출하세요:

👉 **[PetFinder Pawpularity 제출 페이지](https://www.kaggle.com/c/petfinder-pawpularity-score/submit)**

> IOAI 2025 *Weather*(위성영상→강수량) 같은 **이미지 회귀**의 기반 기법입니다. 제출하려면 대회에 Join 되어 있어야 합니다.